# Notebook 9 — Counter-example augmented fine-tuning on SciEntsBank  ·  GPU: T4

Identical to Notebook 4 except that the training set is **augmented with synthetic wrong answers built only from training answers** (`perturb.make_counterexamples`; 20% of the training set): `negated` → `contradictory`, `keyword_salad` → `irrelevant`, `swapped` (answer to a different training question) → `irrelevant`. Checkpoint selection uses only the real validation answers.

The key non-circular check is the **real** `contradictory` class of the official test sets (per-class F1 and confusion matrices), which the synthetic negations never touch.

- Same prompt as the zero-shot run; target = label word (`correct`, `partial`, `contradictory`, `irrelevant`, `non-domain`) + EOS; loss only on the target tokens.
- Identical hyper-parameters to the Mohler runs (LoRA r=16, α=16, all projections; 2 epochs; lr 2e-4; batch 8; fp16; seed 42).
- Prediction from the next-token probabilities of the five labels, as in Notebook 2. Reported: **raw** (argmax), plus **calibrated** (logistic regression fitted on the validation predictions only).

**Timing:** training ≈ 140 min, **resumable** (full state saved twice per epoch to `ckpt/`; a disconnect loses ≤ ~25 min). Re-run all cells to continue. Test scoring (~25 min) is also resumable. **Before running:** upload the updated `llm_utils.py` and `perturb.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
os.makedirs(f'{ROOT}/models', exist_ok=True); os.makedirs(f'{ROOT}/ckpt', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -U bitsandbytes accelerate peft

In [ ]:
import json, gc, time, platform
import numpy as np, pandas as pd, torch, transformers, bitsandbytes, peft
from tqdm.auto import tqdm
import nltk, shutil
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)
from asag_utils import *
from llm_utils import *
from perturb import make_counterexamples

versions = {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
            'bitsandbytes': bitsandbytes.__version__, 'peft': peft.__version__, 'gpu': torch.cuda.get_device_name(0)}
se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
prompts = make_prompts(se, 'scientsbank')
ADAPTER = f'{ROOT}/models/scientsbank_llama3_ftaug'
CKPT = f'{ROOT}/ckpt/scientsbank_llama3_ftaug'
LOG = f'{ROOT}/results/nb9_trainlog.json'
OUT = f'{ROOT}/results/scientsbank_llama3_ftaug.csv'
lp_cols = [f'lp_{l}' for l in LABEL_ORDER]
se.split.value_counts()

## 1. Train (skipped if the adapter from a previous session exists)

In [ ]:
model = None
if os.path.exists(LOG) and os.path.exists(f'{ADAPTER}/adapter_config.json'):
    print('Adapter already trained:', ADAPTER)
else:
    t0 = time.time()
    model, tok = load_hf_4bit(BASE_MODEL)
    cand, _ = candidate_ids(tok, label_candidates(chat=False))
    model, n_trainable = add_lora(model, seed=SEED)
    tr, va = se[se.split == 'train'], se[se.split == 'val']
    aug = make_counterexamples(tr, 'scientsbank', seed=SEED)
    assert not set(aug.group) - set(tr.group), 'augmentation must come from training questions only'
    labels = list(tr.label) + list(aug.target)
    ex = build_examples(tok, [prompts[i] for i in tr.index] + make_prompts(aug, 'scientsbank'), [LABEL_WORDS[l] for l in labels])
    first = {l: c for l, c in zip(LABEL_ORDER, cand)}
    assert all(t[0] == first[l] and t[-1] == tok.eos_token_id for (_, t), l in zip(ex, labels)), 'target tokenisation mismatch'
    print(f'train {len(tr)} + synthetic {len(aug)} {aug.aug_type.value_counts().to_dict()}  val {len(va)}  '
          f'trainable params {n_trainable:,}  max length {max(len(p) + len(t) for p, t in ex)} tokens')

    val_texts = [prompts[i] for i in va.index]
    vy = np.array([LABEL_ORDER.index(l) for l in va.label])
    def evaluate():
        lp = candidate_logprobs(model, tok, val_texts, cand, batch_size=8)
        _, mass, pr = labels_from_logprobs(lp)
        return {'val_nll': float(-lp[np.arange(len(vy)), vy].mean()),       # first label token, as scored at inference
                'val_macro_f1': cls_metrics(va.label.values, pr['5way'])['macro_f1'],
                'val_acc': cls_metrics(va.label.values, pr['5way'])['acc'], 'val_mass': float(mass.mean())}

    torch.cuda.reset_peak_memory_stats()
    hist, best = train_lora(model, tok, ex, evaluate, seed=SEED, ckpt_path=CKPT)
    model.save_pretrained(ADAPTER)
    shutil.rmtree(CKPT, ignore_errors=True)
    hist.update(n_train=len(tr), n_aug=len(aug), n_val=len(va), trainable_params=n_trainable, best=best,
                peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9, train_minutes=(time.time() - t0) / 60)
    json.dump(hist, open(LOG, 'w'), indent=1, default=float)     # written last: marks training as finished
    print(f'training done in {hist["train_minutes"]:.1f} min; best checkpoint at epoch {best["epoch"]:.1f}; '
          f'val macro-F1 {best["val_macro_f1"]:.3f}')

## 2. Score validation + the three test sets (resumable, saves every 256 answers)

In [ ]:
if model is None:
    model, tok = load_hf_4bit(BASE_MODEL, adapter=ADAPTER)
    cand, _ = candidate_ids(tok, label_candidates(chat=False))
PRIORITY = ['test_ua', 'test_uq', 'test_ud', 'val']
done_ids = set(pd.read_csv(OUT).uid) if os.path.exists(OUT) else set()
rows = se[se.split.isin(PRIORITY)]
rows = rows.assign(pr=rows.split.map(PRIORITY.index)).sort_values('pr', kind='stable')
todo = [i for i in rows.index if se.uid[i] not in done_ids]
t0 = time.time()
for start in tqdm(range(0, len(todo), 256)):
    idx = todo[start:start + 256]
    lp = candidate_logprobs(model, tok, [prompts[i] for i in idx], cand, batch_size=8)
    chunk = pd.concat([pd.DataFrame({'uid': se.uid.values[idx]}), pd.DataFrame(lp, columns=lp_cols)], axis=1)
    chunk.to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)
if todo:
    print(f'{1000 * (time.time() - t0) / len(todo):.0f} ms per answer')
del model; gc.collect(); torch.cuda.empty_cache()

## 3. Results

In [ ]:
from sklearn.linear_model import LogisticRegression
label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
z = pd.read_csv(OUT).drop_duplicates('uid')
d = se[['uid', 'split'] + list(label_col.values())].merge(z, on='uid')
assert len(d) == se.split.isin(PRIORITY).sum(), 'scoring incomplete: re-run section 2'
q, mass, raw = labels_from_logprobs(d[lp_cols].values)
feats = np.log(np.clip(q, 1e-12, 1))
fit = (d.split == 'val').values                       # calibration fitted on validation predictions only
new = pd.DataFrame({'uid': d.uid})
for scheme, col in label_col.items():
    new[f'llama3_ftaug_{scheme}'] = raw[scheme]
    clf = LogisticRegression(max_iter=2000).fit(feats[fit], d[col].values[fit])
    new[f'llama3_ftaug_cal_{scheme}'] = clf.predict(feats)
preds = pd.read_csv(f'{ROOT}/results/scientsbank_preds.csv')
preds = preds.drop(columns=[c for c in new.columns if c != 'uid' and c in preds.columns]).merge(new, on='uid', how='left')
preds.to_csv(f'{ROOT}/results/scientsbank_preds.csv', index=False)
print(f'mean valid mass {mass.mean():.3f}')

TESTS = ['test_ua', 'test_uq', 'test_ud']
rows = []
for scheme, col in label_col.items():
    for name in ['llama3_ftaug', 'llama3_ftaug_cal']:
        for t in TESTS:
            dd = preds[preds.split == t]
            rows.append({'scheme': scheme, 'system': name, 'test': t, 'n': len(dd),
                         **summarise(dd[col], dd[f'{name}_{scheme}'], cls_metrics,
                                     groups=None if t == 'test_ua' else dd.group)})
nb9 = pd.DataFrame(rows); nb9.to_csv(f'{ROOT}/results/scientsbank_nb9_summary.csv', index=False)
ref = pd.concat([pd.read_csv(f'{ROOT}/results/scientsbank_nb2_summary.csv'), pd.read_csv(f'{ROOT}/results/scientsbank_nb4_summary.csv')])
both = pd.concat([ref[ref.system.isin(['llama3_instruct_zs_cal', 'llama3_ft', 'llama3_ft_cal'])], nb9])
for metric in ['macro_f1', 'acc']:
    print(metric); display(both.pivot_table(index=['scheme', 'system'], columns='test', values=metric)[TESTS].round(3))
display(nb9[['scheme', 'system', 'test', 'macro_f1', 'macro_f1_ci', 'weighted_f1', 'acc']].round(3))

## 4. Per-class F1 on the REAL test answers, augmented vs. non-augmented (the `contradictory` column is the non-circular check)

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
per_class = []
for t in TESTS:
    dd = preds[preds.split == t]
    for sysname in ['llama3_ft', 'llama3_ftaug']:
        f1 = f1_score(dd.label, dd[f'{sysname}_5way'], labels=LABEL_ORDER, average=None, zero_division=0)
        per_class.append({'test': t, 'system': sysname, **dict(zip(LABEL_ORDER, f1))})
    cm = pd.DataFrame(confusion_matrix(dd.label, dd.llama3_ftaug_5way, labels=LABEL_ORDER),
                      index=[f'gold {l}' for l in LABEL_ORDER], columns=[f'pred {l[:7]}' for l in LABEL_ORDER])
    cm.to_csv(f'{ROOT}/results/scientsbank_confusion_ftaug_{t}.csv')
    print(t, '(augmented)'); display(cm)
per_class = pd.DataFrame(per_class); per_class.to_csv(f'{ROOT}/results/scientsbank_perclass_f1_ftaug.csv', index=False)
display(per_class.round(3))

In [ ]:
import matplotlib.pyplot as plt
h = json.load(open(LOG))
cost = {k: h[k] for k in ['n_train', 'n_aug', 'n_val', 'total_steps', 'sec_per_step', 'train_minutes', 'peak_vram_gb', 'trainable_params']}
cost['best_epoch'] = h['best']['epoch']; print(cost)
json.dump(cost, open(f'{ROOT}/results/nb9_training_cost.json', 'w'), indent=1, default=float)
tr, ev = pd.DataFrame(h['train']), pd.DataFrame(h['eval'])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(tr.step, tr.loss); ax[0].set_title('Training loss (target tokens)'); ax[0].set_xlabel('step')
ax[1].plot(ev.epoch, ev.val_nll, 'o-'); ax[1].set_title('Validation NLL (first label token)'); ax[1].set_xlabel('epoch')
ax[2].plot(ev.epoch, ev.val_macro_f1, 'o-'); ax[2].set_title('Validation macro-F1 (5-way)'); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{ROOT}/results/fig_scientsbank_ftaug_learning_curves.png', dpi=300); plt.show()
json.dump(versions, open(f'{ROOT}/results/env_nb9.json', 'w'), indent=2)